# Laptop Dataset: Cleaning, Feature Engineering & Exploratory Data Analysis

**Author:** Data Engineering Team  
**Objective:** Transform raw, unstructured laptop specifications into a clean, structured dataset for machine learning and pricing analysis.

---

## 1. Setup & Environment Initialisation
Import essential libraries for data manipulation, regular expressions, and statistical visualisations.

In [ ]:
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set visualization aesthetics
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 6)
pd.set_option('display.max_columns', None)

## 2. Load Raw Data & Initial Audit
Load the raw CSV file and audit missing values, structural types, and missing value representations (e.g., `'?'`).

In [ ]:
# Load raw dataset and replace common string placeholders with formal NaN
df = pd.read_csv('laptop_data.csv')
df.replace(r'^\s*\?\s*$', np.nan, regex=True, inplace=True)
df.replace(r'^\s*$', np.nan, regex=True, inplace=True)

print(f"Initial Dataset Shape: {df.shape[0]} rows x {df.shape[1]} columns\n")
df.info()

## 3. Data Cleaning & Feature Engineering Pipeline

We process raw text fields into clean numeric types and high-signal structured features:
- **Numeric Parsing:** Strip units (`GB`, `kg`, `$`) from `Ram`, `Weight`, and `Price`.
- **Display Specs:** Extract `Touchscreen`, `IPS`, and calculate Pixels Per Inch (`PPI`).
- **Processor Specs:** Parse `Clock_Speed_GHz` and categorize `Cpu_Brand`.
- **Graphics Specs:** Clean encoding artifacts and classify GPU `Brand`, `Type`, and `Tier`.
- **Storage Drives:** Split compound memory strings into distinct `SSD_GB`, `HDD_GB`, `Flash_GB`, and `Hybrid_GB` columns.
- **OS Consolidation:** Group fragmented categories into core families (`Windows`, `Mac`, `Linux`, `Chrome OS`).

In [ ]:
# 1. Basic Numeric Cleaning & Imputation
if 'Inches' in df.columns:
    df['Inches'] = pd.to_numeric(df['Inches'], errors='coerce')
    df['Inches'] = df.groupby('TypeName')['Inches'].transform(lambda x: x.fillna(x.median()))

if 'Ram' in df.columns:
    df['Ram'] = pd.to_numeric(df['Ram'].astype(str).str.replace('GB', '', case=False).str.strip(), errors='coerce')

if 'Weight' in df.columns:
    df['Weight'] = pd.to_numeric(df['Weight'].astype(str).str.replace('kg', '', case=False).str.strip(), errors='coerce')

if 'Price' in df.columns and df['Price'].dtype == 'object':
    df['Price'] = pd.to_numeric(df['Price'].astype(str).str.replace(r'[$€,]', '', regex=True).str.strip(), errors='coerce')

# 2. Screen Resolution Features
res_str = df['ScreenResolution'].astype(str)
df['Touchscreen'] = res_str.apply(lambda x: 1 if 'Touchscreen' in x else 0)
df['IPS'] = res_str.apply(lambda x: 1 if 'IPS' in x else 0)
resolution = res_str.str.extract(r'(\d+)x(\d+)')
x_res = resolution[0].astype(float)
y_res = resolution[1].astype(float)
df['PPI'] = (np.sqrt(x_res**2 + y_res**2) / df['Inches']).round(2)

# 3. CPU Features
cpu_str = df['Cpu'].astype(str)
df['Clock_Speed_GHz'] = cpu_str.apply(
    lambda x: float(re.search(r'([\d\.]+)GHz', x).group(1)) if re.search(r'([\d\.]+)GHz', x) else np.nan
)

def fetch_processor(text):
    if 'Intel Core i7' in text: return 'Intel Core i7'
    elif 'Intel Core i5' in text: return 'Intel Core i5'
    elif 'Intel Core i3' in text: return 'Intel Core i3'
    elif 'Intel' in text: return 'Other Intel'
    elif 'AMD' in text: return 'AMD'
    return 'Other'

df['Cpu_Brand'] = cpu_str.apply(fetch_processor)

# 4. GPU Features
def clean_gpu_string(gpu_str):
    if pd.isna(gpu_str): return ""
    gpu_str = re.sub(r'<U\+[0-9A-Fa-f]+>', 'M', str(gpu_str).strip())  # Fix encoding error
    return re.sub(r'GTX\s*(\d+)', r'GTX \1', gpu_str, flags=re.IGNORECASE)  # Fix missing spaces

cleaned_gpu = df['Gpu'].apply(clean_gpu_string)
df['Gpu_Brand'] = cleaned_gpu.apply(lambda x: 'Nvidia' if 'Nvidia' in x else ('Intel' if 'Intel' in x else ('AMD' if 'AMD' in x else 'Other')))
df['Gpu_Type'] = cleaned_gpu.apply(lambda x: 'Integrated' if 'Intel' in x else ('Dedicated' if any(t in x for t in ['GeForce', 'Quadro', 'GTX', 'MX', 'Radeon', 'FirePro']) else 'Integrated'))

def get_gpu_tier(x):
    if 'Intel' in x: return 'Intel Iris (High Integrated)' if 'Iris' in x else 'Intel HD/UHD (Standard Integrated)'
    elif 'Nvidia' in x:
        if 'Quadro' in x: return 'Nvidia Quadro (Workstation)'
        elif any(gtx in x for gtx in ['GTX 1080', 'GTX 1070', 'GTX 1060', 'GTX 980', 'GTX 970', 'SLI']): return 'Nvidia High Gaming'
        elif any(gtx in x for gtx in ['GTX 1050', 'GTX 965', 'GTX 960', 'GTX 950']): return 'Nvidia Mid Gaming'
        else: return 'Nvidia Entry/Casual'
    elif 'AMD' in x:
        if 'FirePro' in x or 'Radeon Pro' in x: return 'AMD Pro/Workstation'
        elif any(rx in x for rx in ['RX 580', 'RX 570', 'RX 560', 'R9']): return 'AMD Gaming'
        else: return 'AMD Entry/Casual'
    return 'Other'

df['Gpu_Tier'] = cleaned_gpu.apply(get_gpu_tier)

# 5. Memory / Storage Features
def parse_memory(mem_str):
    mem_str = str(mem_str).replace('.0', '')
    ssd, hdd, flash, hybrid = 0, 0, 0, 0
    for part in mem_str.split('+'):
        match = re.search(r'(\d+)\s*(GB|TB)\s*(SSD|HDD|Flash Storage|Hybrid)', part.strip(), re.IGNORECASE)
        if match:
            size = int(match.group(1)) * (1024 if match.group(2).upper() == 'TB' else 1)
            dtype = match.group(3).title()
            if 'Ssd' in dtype: ssd += size
            elif 'Hdd' in dtype: hdd += size
            elif 'Flash' in dtype: flash += size
            elif 'Hybrid' in dtype: hybrid += size
    return pd.Series([ssd, hdd, flash, hybrid])

df[['SSD_GB', 'HDD_GB', 'Flash_GB', 'Hybrid_GB']] = df['Memory'].apply(parse_memory)

# 6. Consolidate OS
def clean_os(os_str):
    if pd.isna(os_str): return 'Other / No OS'
    s = str(os_str).strip()
    if s in ['macOS', 'Mac OS X']: return 'Mac'
    elif 'Windows' in s: return 'Windows'
    elif s == 'Linux': return 'Linux'
    elif s == 'Chrome OS': return 'Chrome OS'
    return 'Other / No OS'

df['OpSys'] = df['OpSys'].apply(clean_os)

# 7. Final Cleanup: Drop unparsed raw text columns and reset index
raw_cols = ['ScreenResolution', 'Cpu', 'Gpu', 'Memory']
df.drop(columns=[c for c in raw_cols if c in df.columns], inplace=True)
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)

print("Feature Engineering Complete!")
print(f"Final Clean Dataset Shape: {df.shape[0]} rows x {df.shape[1]} columns")

## 4. Exploratory Data Analysis (EDA) & Visualisations

Now that the dataset is completely cleaned, we visualize key relationships to understand pricing drivers.

### Visual 1: Price Distribution & Target Skewness
Retail prices typically display right-skewness due to premium high-end hardware.

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.histplot(df['Price'], kde=True, color='skyblue')
plt.title('Raw Price Distribution (Right-Skewed)')
plt.xlabel('Price')

plt.subplot(1, 2, 2)
sns.histplot(np.log1p(df['Price']), kde=True, color='teal')
plt.title('Log-Transformed Price (Normalised for ML)')
plt.xlabel('Log(Price)')

plt.tight_layout()
plt.show()

### Visual 2: Impact of GPU Type & CPU Brand on Laptop Price
Comparing average price across processor brands and graphics classifications (`Dedicated` vs. `Integrated`).

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.barplot(data=df, x='Cpu_Brand', y='Price', hue='Gpu_Type', ci=None)
plt.title('Average Price by CPU Brand & GPU Type')
plt.xticks(rotation=30)
plt.ylabel('Mean Price')

plt.subplot(1, 2, 2)
sns.boxplot(data=df, x='Gpu_Type', y='Price', palette='Set2')
plt.title('Price Premium: Integrated vs Dedicated GPUs')

plt.tight_layout()
plt.show()

### Visual 3: Correlation Matrix Across Numeric Features
Evaluating linear relationships between physical hardware features (`Ram`, `SSD_GB`, `PPI`, `Weight`) and laptop price.

In [ ]:
numeric_df = df.select_dtypes(include=[np.number])

plt.figure(figsize=(10, 8))
sns.heatmap(numeric_df.corr(), annot=True, fmt=".2f", cmap='coolwarm', linewidths=0.5)
plt.title('Feature Correlation Heatmap')
plt.show()

## 5. Exporting Clean Dataset
Save the final cleaned DataFrame to a CSV file ready for model training.

In [ ]:
# Export clean dataset
output_filename = 'clean_laptop_data.csv'
df.to_csv(output_filename, index=False)

print(f"Successfully exported clean dataset to {output_filename}")
df.head(10)